# Lab 2 — Agentic RAG: grade, route, loop (EP11)

Lab 1 transformed the question and re-ranked the results — but every question still followed one path: search once, in one library, then answer.
In this lab **the agent decides** (§16.7)

**What you'll do**
1. **Three** knowledge sources over the same book: BM25 · embeddings · **the book's table of contents**
2. **Table-of-contents search** — the model walks down the table of contents level by level: open, go back, or use a section *(beyond the textbook: MemWalker, Chen et al., 2023)*
3. **A CRAG grader** — grade each document: correct / ambiguous / incorrect (§16.5.5)
4. **A router** — the LLM picks the right source for each question (Listing 16.9)
5. **The loop** plan → retrieve → evaluate → generate, with `max_iterations` (Listing 16.10 in plain Python — LangGraph comes in Lab 4)

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §16.5.5, §16.7 — CC BY-SA 4.0
> Table-of-contents search: H. Chen et al., *Walking Down the Memory Maze: Beyond Context Limit through Interactive Reading* (MemWalker), arXiv:2310.05029, 2023 · related: RAPTOR (Sarthi et al., ICLR 2024) · PageIndex (VectifyAI, MIT license)

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **3 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep11-lab2-agentic-rag.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | the same key as Lab 1 (this lab costs under 1 US cent) — how to set up the key, and why we chose the model, is in Lab 1 |
| Packages | `openai`, `python-dotenv`, `pypdf`, `numpy`, `tokenizers` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pypdf>=6" numpy "tokenizers==0.23.3"

## 1. Models and the same library

Gemma 4 26B (Makora) answers · Qwen3 Embedding 8B (Nebius) embeds — as in Lab 1.
The same 141-chunk library + the BM25 we wrote ourselves in EP10 Lab 2

In [ ]:
import os, re, math, json, urllib.request
from collections import Counter
import numpy as np
from openai import OpenAI
from pypdf import PdfReader
from tokenizers import Tokenizer

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0, "calls": 0}

def ask(prompt):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": prompt}])
    COST["chat"] += r.usage.cost
    COST["calls"] += 1
    return r.choices[0].message.content

def embed(texts):
    vectors = []
    for i in range(0, len(texts), 64):
        r = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64],
                                     extra_body=EMBED_EXTRA)
        COST["embed"] += r.usage.cost
        vectors += [d.embedding for d in r.data]
    V = np.array(vectors)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

In [ ]:
def ask_json(prompt, schema):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "reply", "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    COST["calls"] += 1
    return json.loads(r.choices[0].message.content)

def obj(**fields):
    return {"type": "object", "properties": fields,
            "required": list(fields), "additionalProperties": False}
TEXT = {"type": "string"}
def one_of(*values):
    return {"type": "string", "enum": list(values)}

In [ ]:
urllib.request.urlretrieve("https://arxiv.org/pdf/2606.24937v1", "hitchhikers-guide-v1.pdf")
reader = PdfReader("hitchhikers-guide-v1.pdf")
HEADER = "H. Roitman — The Hitchhiker’s Guide to Agentic AI: From Foundations to Systems\n"
pages = [{"page": n, "text": reader.pages[n - 1].extract_text().replace(HEADER, "")}
         for n in range(292, 375)]                      # chapters 15–19
tok = Tokenizer.from_pretrained(MODEL)

def chunk_page(page, W=512, O=64):
    ids = tok.encode(page["text"], add_special_tokens=False).ids
    return [{"page": page["page"], "text": tok.decode(ids[s:s + W])}
            for s in range(0, max(len(ids) - O, 1), W - O)]

chunks = [c for p in pages for c in chunk_page(p)]
E = embed([c["text"] for c in chunks])

def words(text):                                        # BM25 from EP10 Lab 2
    return re.findall(r"\w+", text.lower())
docs = [words(c["text"]) for c in chunks]
N, avgdl = len(docs), sum(len(d) for d in docs) / len(docs)
df = Counter(w for d in docs for w in set(d))
tfs = [Counter(d) for d in docs]
def idf(t):
    n = df.get(t, 0)
    return math.log((N - n + 0.5) / (n + 0.5) + 1)
def bm25_rank(query, k1=1.5, b=0.75):
    scores = [sum(idf(t) * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avgdl))
                  for t in words(query)) for d, tf in zip(docs, tfs)]
    return list(np.argsort(-np.array(scores)))
print(len(chunks), "chunks")

## 2. Sources one and two: BM25 and embeddings

Every source returns the same kind of **documents**: `{"source", "pages", "text"}` — so the agent can use any source the same way

In [ ]:
def as_docs(order, source, k=3):
    return [{"source": source, "pages": [chunks[i]["page"]], "text": chunks[i]["text"]}
            for i in order[:k]]

def search_keyword(query):
    return as_docs(bm25_rank(query), "keyword_search")

def search_vector(query):
    return as_docs(list(np.argsort(-(E @ embed([query])[0]))), "vector_db")

## 3. Source three: the book's table of contents *(beyond the textbook)*

The idea comes from **MemWalker** (Chen et al., 2023): arrange the document as a tree and let the model **walk down the tree** itself — at each level
open a subsection · go back up · or use this section, remembering the path it took (working memory).
MemWalker builds its tree from summaries an LLM writes, and uses a model of about 70B — we use **the table of contents already in the PDF**: only section titles and page numbers, no embeddings.
· RAPTOR (Sarthi et al., 2024) and PageIndex (VectifyAI) use a tree of the document in similar ways

In [ ]:
def walk(outline, depth=0):
    for item in outline:
        if isinstance(item, list):
            yield from walk(item, depth + 1)
        else:
            yield depth, item.title, reader.get_destination_page_number(item) + 1

items = [x for x in walk(reader.outline) if x[0] >= 1 and 292 <= x[2] <= 374]
print(len(items), "entries:", items[:3])

In [ ]:
root = {"title": "The book, pages 292-374", "depth": 0, "start": 292, "end": 374, "children": []}
stack = [root]
for n, (depth, title, start) in enumerate(items):
    end = next((s for d, _, s in items[n + 1:] if d <= depth), 374)   # where the next section starts
    node = {"title": title, "depth": depth, "start": start, "end": max(start, end), "children": []}
    while stack[-1]["depth"] >= depth:
        stack.pop()
    stack[-1]["children"].append(node)
    stack.append(node)
print([c["title"] for c in root["children"]])

In [ ]:
PAGE = {p["page"]: p["text"] for p in pages}
MENU = obj(reasoning=TEXT, action=TEXT)
LEAF = obj(reasoning=TEXT, action=one_of("use", "back"))
def label(n):
    return f'{n["title"]} (pp. {n["start"]}-{n["end"]})'

def search_toc(query, max_steps=10, show=False):
    node, path, read = root, [root], []
    for step in range(1, max_steps + 1):
        if node["children"]:
            menu = "\n".join(f"[{i}] {label(c)}" for i, c in enumerate(node["children"]))
            out = ask_json(f"You are finding the part of a book that answers a question, using its table of contents.\n"
                           f"Question: {query}\nYou are at: {' > '.join(n['title'] for n in path)}\n"
                           f"Already read, did not help: {', '.join(read) or 'nothing'}\n\nSections:\n{menu}\n\n"
                           'Set "action" to the number of the section to open, or to "back" if none fits.', MENU)
        else:
            text = "\n".join(PAGE[p] for p in range(node["start"], node["end"] + 1))
            out = ask_json(f"Question: {query}\nYou opened: {label(node)}\n\n{text}\n\n"
                           'Set "action" to "use" if this text helps answer the question, or "back" if it does not.', LEAF)
        action = out["action"].strip("[] ")
        if show:
            print(f"  step {step}: {node['title'][:40]!r} -> {action}")
        # ✏️ TODO 1: open the child section the model chose and add it to the path
        if action == "use":
            return [{"source": "table_of_contents", "text": text,
                     "pages": list(range(node["start"], node["end"] + 1))}]
        if action.isdigit() and int(action) < len(node["children"]):
            node = ___
            path.append(node)
        else:                                           # "back"
            if not node["children"]:
                read.append(node["title"])
            if len(path) > 1:
                path.pop()
                node = path[-1]
    return []

In [ ]:
found = search_toc("Where does the book compare agent orchestration frameworks in a table?", show=True)
print(found[0]["pages"], found[0]["text"][:200])

## 4. The CRAG grader: do the documents really help?

CRAG (§16.5.5) grades each document: **correct** contains the answer · **ambiguous** related but doesn't answer · **incorrect** not about the question.
If none is correct → the book falls back to web search · ours: the agent **tries another source**

In [ ]:
GRADE = obj(reasoning=TEXT, grade=one_of("correct", "ambiguous", "incorrect"))
def grade(query, doc):
    return ask_json(f"Question: {query}\n\nDocument (pages {doc['pages']}):\n{doc['text']}\n\n"
                    "Grade the document. correct: it contains the answer. "
                    "ambiguous: it is related but does not answer. "
                    "incorrect: it is not about the question.", GRADE)["grade"]

question = ("In Roitman's book The Hitchhiker's Guide to Agentic AI, "
            "what analogy does the RAG chapter use to explain RAG?")
for d in search_vector(question):
    print(d["pages"], grade(question, d))

## 5. The router: match the source to the question (Listing 16.9)

LLM-based routing: the model picks a `source` and writes a `refined_query` suited to it, with its `reasoning`.
The book says it's the most flexible, but it **adds one model call**

In [ ]:
SOURCES = {"keyword_search": search_keyword, "vector_db": search_vector, "table_of_contents": search_toc}
ROUTE = obj(reasoning=TEXT, source=one_of(*SOURCES), refined_query=TEXT)

def route(query, tried=()):
    note = (f"\nAlready tried, found nothing useful: {', '.join(tried)}. Pick another source."
            if tried else "")
    return ask_json(f'Given the query: "{query}"\nDecide which knowledge source to use:\n'
                    "- keyword_search: BM25 over the book's text; best for rare exact terms, acronyms and names\n"
                    "- vector_db: embedding search over the book's text; best for questions phrased in different words from the book\n"
                    "- table_of_contents: walk the book's table of contents; best when the question asks where in the book a topic is covered\n"
                    f"Return the source, a refined query for that source, and your reasoning.{note}", ROUTE)

for q in ["What is PLAID?", "How does the textbook add randomness to retry waits?",
          "Where does the book compare agent orchestration frameworks in a table?"]:
    r = route(q)
    print(f"{r['source']:17} {r['refined_query']!r}")

## 6. The agentic RAG loop (Listing 16.10 in plain Python)

1. **Plan** — split the question into sub-queries, one per thing to look up
2. **Retrieve** — for each sub-query: route → search → grade, keeping only correct documents
3. **Evaluate** — is the accumulated context enough to answer every part?
4. Not enough → loop back to retrieve with the missing sub-queries · enough, **or `max_iterations` reached** → **Generate**

In [ ]:
BOOK = "Roitman, The Hitchhiker's Guide to Agentic AI"
LIBRARY = f"Our library is one book: {BOOK}, pages 292-374."
PLAN = obj(sub_queries={"type": "array", "items": TEXT})
def plan(query):
    return ask_json(f"{LIBRARY}\nSplit this question into the separate things we must look up in the book: "
                    "one sub-query per information need, and only one if it asks for one thing."
                    f"\n\nQuestion: {query}", PLAN)["sub_queries"]

EVAL = obj(reasoning=TEXT, sufficient={"type": "boolean"}, missing={"type": "array", "items": TEXT})
def evaluate(query, docs):
    context = "\n\n".join(f"[pages {d['pages']}]\n{d['text']}" for d in docs) or "(nothing yet)"
    return ask_json(f"Question: {query}\n\nRetrieved context:\n{context}\n\n"
                    "Does this context answer every part of the question? If a part is not answered yet, "
                    "list one sub-query per missing part; do not ask for extra detail.", EVAL)

SYSTEM_PROMPT = """You are a helpful assistant. Answer the question using ONLY
the provided context. If the context does not contain enough information,
say so explicitly. Cite your sources using [Doc N] notation."""
def generate(query, docs):
    context = "\n\n".join(f"[Doc {i+1}] (Source: {BOOK}, Pages: {d['pages']})\n{d['text']}"
                          for i, d in enumerate(docs))
    return ask(f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:")

In [ ]:
def agent(query, max_iterations=3):
    start = COST["calls"]
    sub_queries, docs, tried = plan(query), [], []
    print("plan:", sub_queries)
    # ✏️ TODO 2: keep only the documents the grader calls correct
    for iteration in range(1, max_iterations + 1):
        for sq in sub_queries:
            r = route(sq, tried)
            found = SOURCES[r["source"]](r["refined_query"])
            good = ___
            print(f"  round {iteration}: {r['source']:17} {len(good)}/{len(found)} correct  {sq[:45]!r}")
            if not good:
                tried.append(r["source"])
            docs += good
        # ✏️ TODO 3: leave the loop when evaluate says it's enough; otherwise use the missing sub-queries next round
        check = evaluate(query, docs)
        print(f"  evaluate: sufficient = {check['sufficient']}")
        if ___:
            break
        sub_queries = ___ or sub_queries
    answer = generate(query, docs)
    print(f"  {COST['calls'] - start} model calls")
    return answer

In [ ]:
print(agent("What is PLAID, and how does the textbook add randomness to retry waits?"))

In [ ]:
print(agent("Where does the book compare agent orchestration frameworks in a table?"))

In [ ]:
print(agent("What does the acronym PLAID stand for?"))

In [ ]:
print(agent(question))

In [ ]:
print(COST["calls"], "model calls ·", f"chat ${COST['chat']:.4f} · embed ${COST['embed']:.5f}")

## Summary

| Piece | What we saw in the lab |
|---|---|
| Three sources | BM25 · embeddings · the table of contents — all return the same documents |
| Table of contents *(beyond the textbook, MemWalker)* | Gemma 26B walked from the root to Framework Comparison (pp. 359–360) in 3 steps, with no embeddings |
| CRAG grader | the long question: all three embedding top-3 chunks graded incorrect |
| Router | PLAID → keyword · retry randomness → vector · the framework table → table of contents |
| Plan | a two-part question → two sub-queries → two sources → both answered |
| `max_iterations` | "What does PLAID stand for?" — the book never says → 3 rounds → "the context doesn't contain it" (24 calls) |
| Fallback | the long question: a different path each run, but "library card" every time |

**Exercises**
1. **Knowledge refinement (CRAG):** before generating, have Gemma strip the irrelevant sentences from the correct documents
2. Run `agent("What is PLAID?")` several times — on some runs `evaluate` asks what the name stands for and searches two more rounds. Change the prompt so it finishes in round one, then try the two-part question again: does it still find both parts?
3. **Parallel fan-out:** search every source at once and merge with RRF instead of routing (book §16.7.3) — more model calls or fewer?

**Next — Lab 3:** measuring RAG — write Recall@K, MRR, NDCG and faithfulness ourselves, and measure every system we've built